# LC 901 — Online Stock Span
**Difficulty:** Medium | **Category:** Monotonic Stack
**Pattern:** Decreasing Monotonic Stack with Span Accumulation

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> Each stack entry stores (price, span)
not just the price. When a new price arrives and is >= a stack entry,
we absorb that entry's span into the new span. This collapses
previously resolved runs in O(1) amortized without rescanning history.
</div>

## Official Problem Statement

Design an algorithm that collects daily price quotes for some asset
and returns the span of that asset's price for the current day.

The span of the asset's price in one day is the maximum number of
consecutive days (starting from that day and going backward) for
which the daily price was less than or equal to the price of that day.

Implement `StockSpanner`:
- `StockSpanner()` — initializes the object
- `int next(int price)` — returns the span of the given price

**Constraints:**
- `1 <= price <= 100_000`
- At most `10_000` calls to `next`

## What This Is Actually Asking

Every day you receive one stock price.
You must immediately answer: how many consecutive days (counting
today) has the price been at or below today's price?
This is a streaming problem — you cannot look ahead.
The answer for today depends on how many days you'd have to go back
before finding a price strictly higher than today's.

## Walk Through an Example by Hand

```
Call sequence: next(100), next(80), next(60), next(70),
               next(60), next(75), next(85)

Stack stores (price, span). Start empty.

next(100): stack empty → push (100,1). return 1.
  stack: [(100,1)]

next(80):  80<100 → push (80,1). return 1.
  stack: [(100,1),(80,1)]

next(60):  60<80 → push (60,1). return 1.
  stack: [(100,1),(80,1),(60,1)]

next(70):  70>=60 → pop (60,1), span=1+1=2
           70<80  → push (70,2). return 2.
  stack: [(100,1),(80,1),(70,2)]

next(60):  60<70 → push (60,1). return 1.
  stack: [(100,1),(80,1),(70,2),(60,1)]

next(75):  75>=60 → pop (60,1), span=1+1=2
           75>=70 → pop (70,2), span=2+2=4
           75<80  → push (75,4). return 4.
  stack: [(100,1),(80,1),(75,4)]

next(85):  85>=75 → pop (75,4), span=1+4=5
           85>=80 → pop (80,1), span=5+1=6
           85<100 → push (85,6). return 6.
  stack: [(100,1),(85,6)]

Answers: [1, 1, 1, 2, 1, 4, 6]
```

## The Picture

```
Price timeline:

  100|  ██
   85|  ██                      ██
   80|  ██ ██                ██ ██
   75|  ██ ██             ██ ██ ██
   70|  ██ ██       ██    ██ ██ ██
   60|  ██ ██ ██    ██ ██ ██ ██ ██
        D1  D2  D3  D4  D5  D6  D7
       100  80  60  70  60  75  85
        sp=1  1   1   2   1   4   6

Span = how far back you can look without seeing a higher bar:
  D6 span=4: bars at D3,D4,D5,D6 are all <= 75
  D7 span=6: bars at D2..D7 are all <= 85

Stack key: store (price, span) so absorbed runs collapse.
  Popping (75,4) into (85,6) means we never revisit D3-D6 again.
```

## When To Use This Pattern

- When a streaming problem asks for a look-back count against a
  running max/min, think monotonic stack with span accumulation.
- When popping multiple entries that all satisfy the same condition,
  think: collapse their spans into one entry to avoid future re-scans.
- When the problem is "online" (no lookahead), think stack-based
  incremental state rather than precomputation.
- When each input triggers O(1) amortized work regardless of history
  length, think span compression in the stack.
- When you see "consecutive days with price <= today", think
  StockSpanner / monotonic stack with (value, count) pairs.

## The Approach

Maintain a stack of (price, span) pairs in decreasing order of price.
For each new price, initialize span = 1, then pop all stack entries
whose price is <= the new price, accumulating their spans into span.
Push (new_price, span) and return span.
The trick is that each popped entry's span already encodes all the
previously collapsed days, so we never re-examine old data.

In [1]:
from typing import List  # type hints
from collections import deque  # not needed here but common in design Qs

In [2]:
def test_harness(SpannerClass):
    """
    Replay add(price) sequences and check returned spans.
    Each test: (list_of_prices, list_of_expected_spans)
    """
    tests = [
        # (prices,                        expected_spans)
        ([100,80,60,70,60,75,85],          [1,1,1,2,1,4,6]),
        ([31,41,48,59,79],                 [1,2,3,4,5]),
        ([100,100,100],                    [1,2,3]),
        ([50,30,40,20,10],                 [1,1,2,1,1]),
    ]
    total_passed = 0
    total_tests = len(tests)
    for i, (prices, expected_spans) in enumerate(tests):
        spanner = SpannerClass()
        results = [spanner.next(p) for p in prices]
        status = "PASSED" if results == expected_spans else "FAILED"
        if status == "PASSED":
            total_passed += 1
        print(f"Test {i+1}: {status}")
        if status == "FAILED":
            print(f"  Prices:   {prices}")
            print(f"  Expected: {expected_spans}")
            print(f"  Got:      {results}")
    print(f"\n{total_passed}/{total_tests} tests passed.")

In [7]:
from typing import List  # type hints
##from collections import deque  # not needed here but common in design Qs
class StockSpanner:
    """
    Online stock span calculator.

    Maintains a decreasing monotonic stack of (price, span) pairs.
    Each call to next(price) pops entries with price <= current,
    accumulates their spans, then pushes (price, total_span).

    Amortized O(1) per call; O(n) space for the stack.
    """

    def __init__(self):
        """
        Initialize the stock spanner.
        Stack stores (price, span) pairs.
        """
        self.stack = []    # monotonic decreasing and contains pairs of price and 
  

    def next(self, price: int) -> int:
        """
        Return the span for this price.

        Args:
            price: today's stock price
        Returns:
            number of consecutive days (including today)
            with price <= today's price
        """
        span = 1
        #evictions
        while self.stack and self.stack[-1][0] <= price:
            span += self.stack[-1][1]
            self.stack.pop()
        #Add
        self.stack.append([price, span])

        return span
'''
1
1
1
2
1
4
6
---
1
2
3
Test 1: PASSED
Test 2: PASSED
Test 3: PASSED
Test 4: PASSED

4/4 tests passed.
'''



# --- Debug prints (expected in comments) ---
s = StockSpanner()
print(s.next(100))  # 1
print(s.next(80))   # 1
print(s.next(60))   # 1
print(s.next(70))   # 2
print(s.next(60))   # 1
print(s.next(75))   # 4
print(s.next(85))   # 6

print("---")
s2 = StockSpanner()
print(s2.next(100))  # 1
print(s2.next(100))  # 2  (equal counts!)
print(s2.next(100))  # 3

test_harness(StockSpanner)
'''
def test_harness(SpannerClass):
    """
    Replay add(price) sequences and check returned spans.
    Each test: (list_of_prices, list_of_expected_spans)
    """
    tests = [
        # (prices,                        expected_spans)
        ([100,80,60,70,60,75,85],          [1,1,1,2,1,4,6]),
        ([31,41,48,59,79],                 [1,2,3,4,5]),
        ([100,100,100],                    [1,2,3]),
        ([50,30,40,20,10],                 [1,1,2,1,1]),
    ]
    total_passed = 0
    total_tests = len(tests)
    for i, (prices, expected_spans) in enumerate(tests):
        spanner = SpannerClass()
        results = [spanner.next(p) for p in prices]
        status = "PASSED" if results == expected_spans else "FAILED"
        if status == "PASSED":
            total_passed += 1
        print(f"Test {i+1}: {status}")
        if status == "FAILED":
            print(f"  Prices:   {prices}")
            print(f"  Expected: {expected_spans}")
            print(f"  Got:      {results}")
    print(f"\n{total_passed}/{total_tests} tests passed.")
'''





pass

1
1
1
2
1
4
6
---
1
2
3
Test 1: PASSED
Test 2: PASSED
Test 3: PASSED
Test 4: PASSED

4/4 tests passed.


In [ ]:
# Uncomment and run when solution is ready
# test_harness(StockSpanner)

## Complexity

| Approach              | Time (per call) | Space  |
|-----------------------|-----------------|--------|
| Brute Force (rescan)  | O(n)            | O(n)   |
| Monotonic Stack       | O(1) amortized  | O(n)   |

Each price is pushed and popped at most once across all calls,
giving O(n) total work for n calls → O(1) amortized per call.

## Real World Connection

At Citi, 6,000 endpoint health checks emit a score every minute.
A "span" query answers: for how many consecutive minutes has this
endpoint's error rate been at or below its current reading?
This is exactly StockSpanner applied to a health-score stream.
AWS CloudWatch metric math uses a similar rolling window to compute
"M out of N" alarm logic — the span is the M denominator.
The (value, span) compression means the DE pipeline can process
weeks of telemetry history without storing every prior data point.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra